### <b>Asset definition</b>

In [1]:
from importlib import reload
from helpers import *
import pandas as pd

In [ ]:


ticker_universe = {
    # US equities — broad & factor
    "SPY":  "S&P 500",
    "QQQ":  "Nasdaq 100 (growth/tech)",
    "IWM":  "Russell 2000 (small-cap blend)",
    "AVUV": "Avantis US Small-Cap Value (small-value factor)",
    "MTUM": "iShares MSCI USA Momentum",
    "USMV": "iShares MSCI USA Min Vol",
    
    "VT":  "Global All-Cap (total world market)",

    # International equities
    "VEA":  "Developed Markets ex-US",
    "VWO":  "Emerging Markets",

    # Bonds — duration spectrum
    "VGSH": "Short-Term Treasury (1-3y)",
    "IEF":  "Intermediate Treasury (7-10y)",
    "VGLT": "Long-Term Treasury (20-30y)",
    "TIP":  "TIPS (inflation-linked)",
    "AGG":  "US Aggregate Bond",
    "HYG":  "High Yield Corporate",

    # Commodities & real assets
    "GLD":  "Gold",
    "SLV":  "Silver",
    "GSG":  "Broad Commodities (S&P GSCI)",
    "VNQ":  "US REITs",
}

interest_tickers = [
    "SPY", "QQQ", "VT", "VEA", "VGSH", "IEF", "VGLT", "AGG", "GLD", "TLH", "BTC=F", "AAPL", "TSLA"
]

# Remove near-duplicates (e.g., drop 'AGG' if sleeves present)
interest_tickers = drop_near_duplicates(interest_tickers)

#download data for the tickers in the universe
min_window, max_window = common_window(interest_tickers)

start_date = "2012-01-01"
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")


print(start_date, end_date)

LOOKBACK_YEARS = 4
REBALANCE_FREQ = "MS"
# Covariance configuration
COV_METHOD = "oas"   # 'shrunk' | 'empirical' | 'oas' | 'ewma' | 'factor'
COV_PARAMS = {"jitter": 1e-8}    

for ticker in interest_tickers:
    load_data(ticker, start_date, end_date)

2012-01-01 2026-07-05


$BTC=F: possibly delisted; no price data found  (1d 2026-07-05 -> 2026-07-06) (Yahoo error = "Data doesn't exist for startDate = 1783224000, endDate = 1783310400")

1 Failed download:
['BTC=F']: possibly delisted; no price data found  (1d 2026-07-05 -> 2026-07-06) (Yahoo error = "Data doesn't exist for startDate = 1783224000, endDate = 1783310400")


[Docs](https://pyportfolioopt.readthedocs.io/en/stable/index.html)

### Optimization Notes (PyPortfolioOpt)

- Expected returns use PyPortfolioOpt's `mean_historical_return` on adjusted-close prices (simple returns, annualized).
- Covariance is estimated from returns via PyPortfolioOpt risk models (`sample_cov`, Ledoit–Wolf, OAS, or EWMA depending on `cov_method`).
- Optimizers: `EfficientFrontier.min_volatility()` for min-variance and `EfficientFrontier.max_sharpe(risk_free_rate=rf)` for max-Sharpe.
- Long-only and fully invested constraints are enforced by default.

In [3]:
def build_portfolios(tickers, start, end, lookback_years=LOOKBACK_YEARS, freq=REBALANCE_FREQ, cov_method=None, cov_params=None):
    # Use business-month start to avoid holidays
    # offset start by lookback_years to ensure we have enough data for the first rebalance
    start = pd.Timestamp(start) + pd.DateOffset(years=lookback_years)
    rebalance_dates = pd.date_range(start, end, freq="BMS")

    # default cov settings from globals if not provided
    if cov_method is None:
        cov_method = COV_METHOD
    if cov_params is None:
        cov_params = COV_PARAMS

    mv, ms, mc = {}, {}, {}
    for dt in rebalance_dates:
        as_of = dt.date()
        mv[dt] = min_variance(tickers, as_of=as_of, timeframe_years=lookback_years, cov_method=cov_method, cov_params=cov_params)
        ms[dt] = max_sharpe(tickers, as_of=as_of, timeframe_years=lookback_years, cov_method=cov_method, cov_params=cov_params)
        mc[dt] = pd.Series({t: 1.0 if t == "VT" else 0.0 for t in tickers})

    def to_daily(d):
        df = pd.DataFrame(d).T.sort_index()
        # Start the daily index at the first available weight to avoid leading NaNs
        first = df.index.min()
        daily_idx = pd.bdate_range(first, end)
        return df.reindex(daily_idx, method="ffill")

    return {
        "min_variance": to_daily(mv),
        "max_sharpe":   to_daily(ms),
        "market_cap":   to_daily(mc),
    }

In [4]:
portfolios = build_portfolios(interest_tickers, start_date, end_date, cov_method=COV_METHOD, cov_params=COV_PARAMS)

portfolios
# portfolios["min_variance"], portfolios["max_sharpe"], portfolios["market_cap"]
# each is a DataFrame: index=business days, columns=tickers, values=weights


/Users/giacomomaggiore/Desktop/coding/leverage-api-api/.venv/lib/python3.12/site-packages/pypfopt/base_optimizer.py:307: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  self._opt.solve(
/Users/giacomomaggiore/Desktop/coding/leverage-api-api/.venv/lib/python3.12/site-packages/pypfopt/base_optimizer.py:307: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  self._opt.solve(


{'min_variance':                 SPY  QQQ   VT  VEA     VGSH  IEF  VGLT  AGG  GLD  TLH  BTC=F  \
 2016-01-01  0.01570  0.0  0.0  0.0  0.98322  0.0   0.0  0.0  0.0  0.0    0.0   
 2016-01-04  0.01570  0.0  0.0  0.0  0.98322  0.0   0.0  0.0  0.0  0.0    0.0   
 2016-01-05  0.01570  0.0  0.0  0.0  0.98322  0.0   0.0  0.0  0.0  0.0    0.0   
 2016-01-06  0.01570  0.0  0.0  0.0  0.98322  0.0   0.0  0.0  0.0  0.0    0.0   
 2016-01-07  0.01570  0.0  0.0  0.0  0.98322  0.0   0.0  0.0  0.0  0.0    0.0   
 ...             ...  ...  ...  ...      ...  ...   ...  ...  ...  ...    ...   
 2026-06-29  0.01192  0.0  0.0  0.0  0.98808  0.0   0.0  0.0  0.0  0.0    0.0   
 2026-06-30  0.01192  0.0  0.0  0.0  0.98808  0.0   0.0  0.0  0.0  0.0    0.0   
 2026-07-01  0.01528  0.0  0.0  0.0  0.98472  0.0   0.0  0.0  0.0  0.0    0.0   
 2026-07-02  0.01528  0.0  0.0  0.0  0.98472  0.0   0.0  0.0  0.0  0.0    0.0   
 2026-07-03  0.01528  0.0  0.0  0.0  0.98472  0.0   0.0  0.0  0.0  0.0    0.0   
 
          

In [5]:
portfolios["min_variance"]

,SPY,QQQ,VT,VEA,VGSH,IEF,VGLT,AGG,GLD,TLH,BTC=F,AAPL,TSLA
2016-01-01,0.01570,0.0,0.0,0.0,0.98322,0.0,0.0,0.0,0.0,0.0,0.0,0.00108,0.0
2016-01-04,0.01570,0.0,0.0,0.0,0.98322,0.0,0.0,0.0,0.0,0.0,0.0,0.00108,0.0
2016-01-05,0.01570,0.0,0.0,0.0,0.98322,0.0,0.0,0.0,0.0,0.0,0.0,0.00108,0.0
2016-01-06,0.01570,0.0,0.0,0.0,0.98322,0.0,0.0,0.0,0.0,0.0,0.0,0.00108,0.0
2016-01-07,0.01570,0.0,0.0,0.0,0.98322,0.0,0.0,0.0,0.0,0.0,0.0,0.00108,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-29,0.01192,0.0,0.0,0.0,0.98808,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0
2026-06-30,0.01192,0.0,0.0,0.0,0.98808,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0
2026-07-01,0.01528,0.0,0.0,0.0,0.98472,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0
2026-07-02,0.01528,0.0,0.0,0.0,0.98472,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0


In [ ]:
portfolios["max_sharpe"]

,SPY,QQQ,VT,VEA,VGSH,IEF,VGLT,AGG,GLD,TLH,BTC=F,AAPL,TSLA
2016-01-01,1.791650e-02,0.101947,7.187952e-15,1.180190e-14,0.622316,6.246768e-13,3.954672e-13,1.662436e-01,3.410983e-14,7.391282e-02,0.000000,8.501977e-16,1.766451e-02
2016-01-04,1.791650e-02,0.101947,7.187952e-15,1.180190e-14,0.622316,6.246768e-13,3.954672e-13,1.662436e-01,3.410983e-14,7.391282e-02,0.000000,8.501977e-16,1.766451e-02
2016-01-05,1.791650e-02,0.101947,7.187952e-15,1.180190e-14,0.622316,6.246768e-13,3.954672e-13,1.662436e-01,3.410983e-14,7.391282e-02,0.000000,8.501977e-16,1.766451e-02
2016-01-06,1.791650e-02,0.101947,7.187952e-15,1.180190e-14,0.622316,6.246768e-13,3.954672e-13,1.662436e-01,3.410983e-14,7.391282e-02,0.000000,8.501977e-16,1.766451e-02
2016-01-07,1.791650e-02,0.101947,7.187952e-15,1.180190e-14,0.622316,6.246768e-13,3.954672e-13,1.662436e-01,3.410983e-14,7.391282e-02,0.000000,8.501977e-16,1.766451e-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-29,5.253592e-16,0.048113,0.000000e+00,3.128244e-16,0.892700,1.706244e-15,0.000000e+00,1.965706e-15,5.918713e-02,0.000000e+00,0.000000,1.413265e-15,1.972032e-15
2026-06-30,5.253592e-16,0.048113,0.000000e+00,3.128244e-16,0.892700,1.706244e-15,0.000000e+00,1.965706e-15,5.918713e-02,0.000000e+00,0.000000,1.413265e-15,1.972032e-15
2026-07-01,1.068586e-13,0.049702,7.656827e-14,0.000000e+00,0.892025,4.251822e-14,1.179298e-13,4.244097e-14,5.243592e-02,9.635245e-14,0.005837,1.910261e-14,8.198145e-14
2026-07-02,1.068586e-13,0.049702,7.656827e-14,0.000000e+00,0.892025,4.251822e-14,1.179298e-13,4.244097e-14,5.243592e-02,9.635245e-14,0.005837,1.910261e-14,8.198145e-14
